<a href="https://colab.research.google.com/github/antoniovfonseca/agentic-ai-global-lulc/blob/main/MCP_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 1. Environment Setup

Install the official `mcp` (Model Context Protocol) SDK (Software Development Kit) and the `numpy` library for numerical processing.

In [2]:
# 1. Install the Model Context Protocol SDK and numpy
!pip install -q mcp numpy

## 2. Imports and Server Initialization

Import the libraries and initialize the FastMCP server instance.

In [3]:
import numpy as np
from mcp.server.fastmcp import FastMCP

mcp = FastMCP("My Colab MCP")

## 3. Tool Registration

Define a function decorated with `@mcp.tool()`. This example uses `numpy` to calculate statistics (mean and standard deviation) from a list of numbers, demonstrating how to expose Python logic to the MCP server.

In [39]:
import numpy as np

@mcp.tool()
def calculate_statistics(data: list[float]) -> str:
    """
    Computes the statistical mean and standard deviation for a sequence of numbers.

    Parameters
    ----------
    data : list of float
        A sequence of numerical values to be processed.

    Returns
    -------
    str
        A formatted string containing the calculated mean and standard deviation.
    """
    array_data = np.array(data)
    mean_val = np.mean(array_data)
    std_val = np.std(array_data)

    return f"Mean: {mean_val:.2f}, Standard Deviation: {std_val:.2f}"

## 4. Local Tool Verification

Test the registered tool by invoking the function directly with sample data. This ensures the logic behaves as expected before starting the server process.

In [6]:
# 1. Define a sample list of floats for testing (Mock Data)
sample_data = [10.5, 22.3, 33.1, 45.0, 12.8]

# 2. Call the function directly to verify the output
result = calculate_statistics(sample_data)

# 3. Print the formatted result to the console
print(f"Test Result: {result}")

Test Result: Mean: 24.74, Standard Deviation: 12.90


## 5. Running the Server (Threaded Approach)

Since Google Colab controls the main event loop, I execute the MCP server in a separate thread. This prevents the `asyncio` conflict and allows the server to run in the background.

In [7]:
import threading

# 1. Define a wrapper function to run the server logic
def run_mcp_server():
    try:
        # mcp.run() creates its own event loop, which works fine inside a new thread
        mcp.run()
    except Exception as e:
        print(f"\nServer error: {e}")

# 2. Create a new thread targeting the wrapper function
# daemon=True ensures the thread closes when the notebook kernel shuts down
server_thread = threading.Thread(
    target=run_mcp_server,
    daemon=True
)

# 3. Start the server thread
print("Starting MCP Server in a background thread...")
server_thread.start()

# 4. Confirmation message
print("Server is running! You can now continue using other cells.")

Starting MCP Server in a background thread...
Server is running! You can now continue using other cells.


## 6. Installing and Configuring the Gemini LLM

Here I install the official Gemini SDK to act as the brain of our Agentic AI.

In [8]:
!pip install -q google-genai

In [10]:
from google import genai
from google.colab import userdata

# 1. Retrieve the API key securely from Colab's secret manager
api_key = userdata.get('GEMINI_API_KEY')

# 2. Initialize the connection to the Gemini LLM
gemini_client = genai.Client(api_key=api_key)
print("Gemini LLM successfully connected!")

Gemini LLM successfully connected!


## 7. The Agentic Loop: Triggering Function Calls

Here I create a simple orchestrator. I send a natural language prompt to Gemini and provide our `calculate_statistics` tool. We will observe how the LLM decides to format a tool call instead of generating regular text.

### Version 1
The LLM did not use my function to answer my question.

In [12]:
# 1. The user's request in natural (unstructured) language
user_prompt = "Hi! Can you calculate the mean and standard deviation of these numbers for me? The data is 10.5, 22.3, 33.1, 45.0, and 12.8."

# 2. The Orchestrator sends the context to the LLM, attaching the tool
response = gemini_client.models.generate_content(
    model='gemini-2.5-flash',
    contents=user_prompt,
    config={
        "tools": [calculate_statistics], # Connecting the tool's "manual" to the brain
    }
)

# 3. Inspecting the Agent's decision (Function Calling)
if response.function_calls:
    print("The Agent decided it needs to use a tool!\n")
    for tool_call in response.function_calls:
        print(f"Tool chosen by the LLM: {tool_call.name}")
        print(f"Arguments extracted from the text: {tool_call.args}")
else:
    print("The Agent responded directly with text:")
    print(response.text)

The Agent responded directly with text:
The mean of the numbers is 24.74 and the standard deviation is 12.90.


### Version 2
Exceeded curret quota.

Models must be Text-out models (?)

In [15]:
# 1. The user's request in natural (unstructured) language
user_prompt = "Hi! Can you calculate the mean and standard deviation of these numbers for me? The data is 10.5, 22.3, 33.1, 45.0, and 12.8."

# 2. The Orchestrator sends the context and FORCES the use of a tool
response = gemini_client.models.generate_content(
    model='gemini-2.5-flash',
    contents=user_prompt,
    config={
        "tools": [calculate_statistics],
        # Tool Forcing: Obriga o LLM a acionar uma ferramenta listada
        "tool_config": {"function_calling_config": {"mode": "ANY"}}
    }
)

# 3. Inspecting the Agent's decision (Function Calling)
if response.function_calls:
    print("The Agent decided it needs to use a tool!\n")
    for tool_call in response.function_calls:
        print(f"Tool chosen by the LLM: {tool_call.name}")
        print(f"Arguments extracted from the text: {tool_call.args}")
else:
    print("🤖 The Agent responded directly with text:")
    print(response.text)

ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash\nPlease retry in 47.25910834s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeTier', 'quotaDimensions': {'location': 'global', 'model': 'gemini-2.5-flash'}, 

### Version 3

In [23]:
# List all models available for your API Key to avoid 404 errors
print("Listing available models...")
for model in gemini_client.models.list():
    print(f"Model ID: {model.name} - Supported Actions: {model.supported_actions}")

Listing available models...
Model ID: models/gemini-2.5-flash - Supported Actions: ['generateContent', 'countTokens', 'createCachedContent', 'batchGenerateContent']
Model ID: models/gemini-2.5-pro - Supported Actions: ['generateContent', 'countTokens', 'createCachedContent', 'batchGenerateContent']
Model ID: models/gemini-2.0-flash - Supported Actions: ['generateContent', 'countTokens', 'createCachedContent', 'batchGenerateContent']
Model ID: models/gemini-2.0-flash-001 - Supported Actions: ['generateContent', 'countTokens', 'createCachedContent', 'batchGenerateContent']
Model ID: models/gemini-2.0-flash-lite-001 - Supported Actions: ['generateContent', 'countTokens', 'createCachedContent', 'batchGenerateContent']
Model ID: models/gemini-2.0-flash-lite - Supported Actions: ['generateContent', 'countTokens', 'createCachedContent', 'batchGenerateContent']
Model ID: models/gemini-2.5-flash-preview-tts - Supported Actions: ['countTokens', 'generateContent']
Model ID: models/gemini-2.5-pro-

In [38]:
# List only models that support text generation (Agentic capabilities)
print("Listing available text-out models...")

for model in gemini_client.models.list():
    # Filter: Check if the model supports generating content (text/reasoning)
    if 'generateContent' in model.supported_actions:
        print(f"Model ID: {model.name} - Supported Actions: {model.supported_actions}")

Listing available text-out models...
Model ID: models/gemini-2.5-flash - Supported Actions: ['generateContent', 'countTokens', 'createCachedContent', 'batchGenerateContent']
Model ID: models/gemini-2.5-pro - Supported Actions: ['generateContent', 'countTokens', 'createCachedContent', 'batchGenerateContent']
Model ID: models/gemini-2.0-flash - Supported Actions: ['generateContent', 'countTokens', 'createCachedContent', 'batchGenerateContent']
Model ID: models/gemini-2.0-flash-001 - Supported Actions: ['generateContent', 'countTokens', 'createCachedContent', 'batchGenerateContent']
Model ID: models/gemini-2.0-flash-lite-001 - Supported Actions: ['generateContent', 'countTokens', 'createCachedContent', 'batchGenerateContent']
Model ID: models/gemini-2.0-flash-lite - Supported Actions: ['generateContent', 'countTokens', 'createCachedContent', 'batchGenerateContent']
Model ID: models/gemini-2.5-flash-preview-tts - Supported Actions: ['countTokens', 'generateContent']
Model ID: models/gemini

In [31]:
import time

# Wait 5 seconds just to ensure any global network cooldown
time.sleep(5)

user_prompt = "Hi! Can you calculate the mean and standard deviation of these numbers? Data: [10.5, 22.3, 33.1, 45.0, 12.8]"

try:
    # Using the Gemini 2.5 Flash Lite model confirmed in your dashboard
    response = gemini_client.models.generate_content(
        model='gemini-2.5-flash-lite',
        contents=user_prompt,
        config={
            "tools": [calculate_statistics],
            "tool_config": {"function_calling_config": {"mode": "ANY"}}
        }
    )

    if response.function_calls:
        print("Success: Tool call identified")
        for tool_call in response.function_calls:
            print(f"Tool name: {tool_call.name}")
            print(f"Arguments: {tool_call.args}")
    else:
        print(f"Response: {response.text}")

except Exception as e:
    print(f"Execution failed: {e}")

Success: Tool call identified
Tool name: calculate_statistics
Arguments: {'data': [10.5, 22.3, 33.1, 45, 12.8]}


### Version 4

In [40]:
import time
from datetime import datetime

def log_event(event_name: str) -> None:
    """
    Logs a given event string to the standard output with a timestamp prefix.

    Parameters
    ----------
    event_name : str
        The descriptive string representing the event or action to be logged.

    Returns
    -------
    None
        This function does not return any value.
    """
    timestamp = datetime.now().strftime("%H:%M:%S")
    print(f"[{timestamp}] {event_name}")

# --- MITIGATION: GLOBAL RATE LIMIT COOLDOWN ---
# Protects against RPM exhaustion if this script/cell is executed multiple times in a row.
print("Applying global network cooldown (5 seconds) before starting...")
time.sleep(5)

# Define the user prompt
user_prompt = "Calculate the mean and standard deviation for: [10.5, 22.3, 33.1, 45.0, 12.8]"

try:
    print("==================================================")
    log_event("STARTING AGENTIC LOOP")

    # 1. First Turn: Forcing tool usage
    log_event("Host -> LLM: Sending prompt and enforcing tool protocol")
    response = gemini_client.models.generate_content(
        model='gemini-3.1-flash-lite-preview',
        contents=user_prompt,
        config={
            "tools": [calculate_statistics],
            "tool_config": {"function_calling_config": {"mode": "ANY"}}
        }
    )

    # 2. Protocol Check
    if response.function_calls:
        tool_call = response.function_calls[0]
        log_event(f"LLM -> Host: Tool request detected ('{tool_call.name}')")

        # 3. Server Execution
        log_event("Host -> Server: Executing local function")
        tool_result = calculate_statistics(**tool_call.args)
        log_event(f"Server -> Host: Execution complete. Raw result: {tool_result}")

        # --- MITIGATION: INTERNAL RATE LIMIT COOLDOWN ---
        log_event("Host: Applying safety cooldown to prevent internal RPM limits...")
        time.sleep(5)

        # 4. Final Synthesis
        log_event("Host -> LLM: Sending exact tool data for final synthesis")
        final_response = gemini_client.models.generate_content(
            model='gemini-3.1-flash-lite-preview',
            contents=[
                user_prompt,
                response.candidates[0].content,
                {
                    "role": "user",
                    "parts": [{
                        "function_response": {
                            "name": tool_call.name,
                            "response": {"output": str(tool_result)}
                        }
                    }]
                }
            ]
        )

        log_event("LLM -> Host: Final response generated")
        print("==================================================\n")

        print("--- Final Agent Response (via MCP) ---")
        print(final_response.text)

    else:
        log_event("Protocol Failure: No tool triggered.")

except Exception as e:
    print(f"Execution error: {e}")

Applying global network cooldown (5 seconds) before starting...
[18:38:42] STARTING AGENTIC LOOP
[18:38:42] Host -> LLM: Sending prompt and enforcing tool protocol
[18:38:49] LLM -> Host: Tool request detected ('calculate_statistics')
[18:38:49] Host -> Server: Executing local function
[18:38:49] Server -> Host: Execution complete. Raw result: Mean: 24.74, Standard Deviation: 12.90
[18:38:49] Host: Applying safety cooldown to prevent internal RPM limits...
[18:38:54] Host -> LLM: Sending exact tool data for final synthesis
[18:38:57] LLM -> Host: Final response generated

--- Final Agent Response (via MCP) ---
To calculate the mean and standard deviation for the dataset **[10.5, 22.3, 33.1, 45.0, 12.8]**:

### 1. Calculate the Mean ($\mu$)
The mean is the sum of the numbers divided by the count ($n = 5$):
$$\text{Mean} = \frac{10.5 + 22.3 + 33.1 + 45.0 + 12.8}{5} = \frac{123.7}{5} = \mathbf{24.74}$$

### 2. Calculate the Standard Deviation ($\sigma$)
To find the population standard dev